# MoViNet-A0: Hiểu Kiến Trúc & Khởi Tạo Model

> **Update:** Package `Atze00/MoViNet-pytorch` đã có sẵn pretrained weights cho A0 (Kinetics-600),
> download tự động qua `pretrained=True`. Không cần convert TF checkpoint thủ công.

**Mục tiêu notebook:**
1. Hiểu kiến trúc MoViNet-A0 (causal convolutions, stream buffer)
2. Load pretrained model và kiểm tra forward pass
3. Hiểu cách streaming inference hoạt động
4. Lưu weights ra `models/video/movinet_a0_k600.pt` để dùng cho fine-tune

---
## Section 1: Kiến Trúc MoViNet-A0

### Tại sao MoViNet phù hợp realtime hơn?

| Model | Temporal | Stream | Params | Min Frames | Realtime |
|---|---|---|---|---|---|
| R3D-18 | 3D Conv | ❌ | 33M | 8 | ❌ |
| S3D | Factored 3D | ❌ | 8.3M | **16** | ❌ |
| MViT-V2-S | Attention | ❌ | 34M | 16 | ❌ |
| **MoViNet-A0** | **Causal 2+1D** | **✅** | **3.75M** | **1** | **✅** |

### Causal Convolution

```
Standard 3D Conv:  [t-k,...,t,...,t+k] → nhìn quá khứ VÀ tương lai → offline only
Causal 2+1D Conv:  [t-k,...,t]         → chỉ nhìn quá khứ → streaming OK
    ↪ 2D spatial conv: xử lý H × W tại từng frame
    ↪ 1D temporal conv (causal): kết nối các frame theo thứ tự
```

### Stream Buffer

```
Video: [f0, f1, f2, ..., f29]  (30 frames)

Không dùng stream buffer (offline):
  model([f0...f29]) → 1 inference, cần toàn bộ video

Dùng stream buffer (causal=True):
  model([f0...f5])   → buffer lưu temporal state
  model([f6...f11])  → đọc buffer + xử lý clip mới
  model([f12...f17]) → đọc buffer + xử lý clip mới
  model([f18...f23]) → đọc buffer + xử lý clip mới
  model([f24...f29]) → kết quả cuối cùng (có context 30 frames!)
  model.clean_activation_buffers()  ← reset trước video tiếp theo
```

### Kiến Trúc MoViNet-A0 chi tiết

```
Input: [B, C, T, H, W] = [batch, 3, n_frames, 172, 172]
  ↓
  Stem: Conv2+1D causal (3→16, stride=2) → [B,16,T,86,86]
  ↓
  MoViNet Blocks (5×):
    each = BottleneckResidual(
      Expand: Conv2+1D (pointwise)
      Depthwise: Conv2+1D causal
      Squeeze-Excitation: avg pool → FC → FC → sigmoid
      Project: Conv2+1D (pointwise)
    )
  ↓
  Head: Conv2+1D → AvgPool → FC 600
  ↓
Output: [B, 600]  (Kinetics-600 logits, pretrained)
```

---
## Section 2: Cài Đặt & Load Pretrained Model

In [ ]:
import sys

import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("XPU available:", torch.xpu.is_available())

# Cài movinets nếu chưa có
try:
    import movinets

    print("movinets: already installed")
except ImportError:
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "git+https://github.com/Atze00/MoViNet-pytorch.git",
            "--quiet",
        ]
    )

    print("movinets: installed")

In [ ]:
from movinets import MoViNet
from movinets.config import _C

# Load MoViNet-A0-Stream (causal=True) với Kinetics-600 weights
# Weights sẽ được download tự động về ~/.cache/torch/hub/checkpoints/
print("Loading MoViNet-A0-Stream pretrained weights (Kinetics-600)...")
model = MoViNet(_C.MODEL.MoViNetA0, causal=True, pretrained=True)
model.eval()
print("Load complete!")

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total:>10,} ({total/1e6:.2f}M)")
print(f"Trainable parameters: {trainable:>10,} ({trainable/1e6:.2f}M)")

In [ ]:
# In kiến trúc model
print(model)

---
## Section 3: Kiểm Tra Forward Pass

In [ ]:
import torch
import torch.nn.functional as F

# Test 1: Single clip inference
dummy = torch.zeros(1, 3, 6, 172, 172)  # batch=1, RGB, 6 frames, 172×172
with torch.no_grad():
    out = model(dummy)
print(f"Single clip [1,3,6,172,172] → output: {out.shape}")
print(f"Probabilities sum: {F.softmax(out, dim=1).sum().item():.4f}")  # Should be ~1.0

# Test 2: Stream buffer
model.clean_activation_buffers()
print("clean_activation_buffers(): OK")

In [ ]:
# Streaming inference demo: 5 clips × 6 frames = 30 frames video
import time

N_CLIPS = 5
N_CLIP_FRAMES = 6
video = torch.randn(1, 3, N_CLIPS * N_CLIP_FRAMES, 172, 172)

model.clean_activation_buffers()
t0 = time.perf_counter()

clip_logits = []
with torch.no_grad():
    for j in range(N_CLIPS):
        clip = video[:, :, j * N_CLIP_FRAMES : (j + 1) * N_CLIP_FRAMES]
        logits = model(clip)
        clip_logits.append(logits)
        top_class = logits.argmax(dim=1).item()
        top_prob = F.softmax(logits, dim=1).max().item()
        print(
            f"  Clip {j+1}/{N_CLIPS}: top-1 class={top_class:3d}, prob={top_prob:.3f}"
        )

elapsed = (time.perf_counter() - t0) * 1000
model.clean_activation_buffers()

print(f"\nTotal streaming time: {elapsed:.1f}ms ({elapsed/N_CLIPS:.1f}ms/clip)")
print(f"Final logits shape: {clip_logits[-1].shape}")

In [ ]:
# Kiểm tra top-5 classes Kinetics-600 trên input ngẫu nhiên
test = torch.randn(1, 3, 6, 172, 172)
model.clean_activation_buffers()
with torch.no_grad():
    logits = model(test)
probs = F.softmax(logits, dim=1)[0]
top5 = probs.topk(5)
print("Top-5 Kinetics-600 predictions (random input, no meaning):")
for val, idx in zip(top5.values, top5.indices):
    print(f"  class {idx.item():3d}: {val.item():.4f}")
model.clean_activation_buffers()

---
## Section 4: Lưu Weights Cho Fine-tune

In [ ]:
from pathlib import Path

output_dir = Path("models/video")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "movinet_a0_k600.pt"

torch.save(model.state_dict(), output_path)
size_mb = output_path.stat().st_size / 1e6
print(f"Saved: {output_path.absolute()}")
print(f"Size:  {size_mb:.1f} MB")

# Verify reload
model_reload = MoViNet(_C.MODEL.MoViNetA0, causal=True, pretrained=False)
model_reload.load_state_dict(torch.load(output_path, weights_only=True))
model_reload.eval()
print("Reload verify: OK")

---
## Section 5: Thống Kê & Benchmark Nhanh

In [ ]:
import time

# Benchmark CPU latency per clip
clip = torch.randn(1, 3, 6, 172, 172)
WARMUP, ITERS = 5, 20

model.eval()
with torch.no_grad():
    for _ in range(WARMUP):
        model.clean_activation_buffers()
        _ = model(clip)

    times = []
    for _ in range(ITERS):
        model.clean_activation_buffers()
        t0 = time.perf_counter()
        _ = model(clip)
        times.append((time.perf_counter() - t0) * 1000)

import statistics

print("CPU Inference per clip (6 frames @ 172x172):")
print(f"  Mean:   {statistics.mean(times):.1f}ms")
print(f"  Median: {statistics.median(times):.1f}ms")
print(f"  P95:    {sorted(times)[int(0.95*ITERS)]:.1f}ms")
print(f"\n→ Estimated streaming FPS: {1000/statistics.mean(times)*6:.1f} frames/sec")

In [ ]:
# Benchmark XPU nếu có
if torch.xpu.is_available():
    device = torch.device("xpu")
    model_xpu = model.to(device)
    clip_xpu = clip.to(device)

    with torch.no_grad():
        for _ in range(WARMUP):
            model_xpu.clean_activation_buffers()
            _ = model_xpu(clip_xpu)
        torch.xpu.synchronize()

        times_xpu = []
        for _ in range(ITERS):
            model_xpu.clean_activation_buffers()
            t0 = time.perf_counter()
            _ = model_xpu(clip_xpu)
            torch.xpu.synchronize()
            times_xpu.append((time.perf_counter() - t0) * 1000)

    print("XPU (Arc A770) Inference per clip:")
    print(f"  Mean:   {statistics.mean(times_xpu):.1f}ms")
    print(f"  Median: {statistics.median(times_xpu):.1f}ms")
    print(f"→ Speedup vs CPU: {statistics.mean(times)/statistics.mean(times_xpu):.1f}×")
else:
    print("XPU not available")

---
## Checklist

- [ ] `movinets` package cài thành công
- [ ] MoViNet-A0 pretrained weights load OK (3.75M params)
- [ ] Forward pass single clip: `[1,3,6,172,172] → [1,600]` OK
- [ ] Streaming demo 5 clips chạy không lỗi
- [ ] `models/video/movinet_a0_k600.pt` đã lưu
- [ ] Reload verify OK
- [ ] CPU benchmark hoàn tất
- [ ] XPU benchmark hoàn tất (nếu có)

## Next Steps

```bash
# Sau khi notebook chạy xong:
python -m src.training.scripts.movinet describe
python -m src.training.scripts.movinet fine-tune --config configs/models/movinet_a0.yaml --dry-run
```